<a href="https://colab.research.google.com/github/Tuqaghazawi/khcc-patient-intake/blob/main/Copy_of_KHCC_Patient_Intake_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 KHCC Outpatient Oncology — Automated Patient Intake Processing System

**Course:** CCI Prompt Engineering & Clinical AI Development  
**Institution:** King Hussein Cancer Center (KHCC), Amman, Jordan  
**Purpose:** Automate the daily 2–3 hour manual intake review performed by data entry clerks in the outpatient oncology clinic.

---

### Notebook Sections
1. Setup & Imports
2. `PatientIntake` Data Model
3. Clinical Alert Logic (Lab Flags, Risk Stratification, Drug Interactions)
4. Patient Dataset (≥15 patients)
5. Group-Level Statistical Analysis
6. CSV Export (3 files)
7. Simulated Email Summary
8. Critical Reflection
9. Stretch Challenge (Email Report Generator + Multi-day Trends + Deployment Proposal)

---
## Section 1 — Setup & Imports

In [18]:
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple
from datetime import date, datetime, timedelta
import json
import os
import warnings
warnings.filterwarnings('ignore')

# For output directory
OUTPUT_DIR = 'khcc_intake_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Imports successful')
print(f'📁 Output directory: {OUTPUT_DIR}/')

✅ Imports successful
📁 Output directory: khcc_intake_output/


---
## Section 2 — `PatientIntake` Data Model

The `PatientIntake` dataclass models all fields collected during outpatient oncology intake:
- **Demographics**: patient ID, name (initials only for PHI hygiene in demo), age, sex, diagnosis
- **Lab values**: CBC (WBC, Hgb, Plt, ANC), metabolic panel (Cr, Na, K, Mg, ALT, AST, total bilirubin)
- **Performance status**: ECOG score
- **Medications**: current medication list (for interaction checking)
- **Metadata**: intake date, clinician notes

In [19]:
@dataclass
class LabValues:
    """Structured container for lab results with clinical reference ranges."""
    # Complete Blood Count
    wbc: Optional[float] = None        # x10^9/L  (normal 4.5–11.0)
    hemoglobin: Optional[float] = None # g/dL     (normal M:13.5–17.5, F:12.0–15.5)
    platelets: Optional[float] = None  # x10^9/L  (normal 150–400)
    anc: Optional[float] = None        # x10^9/L  (normal 1.8–8.0)

    # Metabolic Panel
    creatinine: Optional[float] = None # mg/dL    (normal 0.7–1.3)
    sodium: Optional[float] = None     # mEq/L    (normal 136–145)
    potassium: Optional[float] = None  # mEq/L    (normal 3.5–5.0)
    magnesium: Optional[float] = None  # mg/dL    (normal 1.7–2.2)
    alt: Optional[float] = None        # U/L      (normal 7–56)
    ast: Optional[float] = None        # U/L      (normal 10–40)
    total_bilirubin: Optional[float] = None  # mg/dL (normal 0.2–1.2)


@dataclass
class PatientIntake:
    """Complete patient intake record for KHCC outpatient oncology."""

    # --- Demographics ---
    patient_id: str
    initials: str                      # PHI-reduced: initials only
    age: int
    sex: str                           # 'M' or 'F'
    diagnosis: str                     # Primary oncologic diagnosis
    diagnosis_stage: Optional[str] = None
    ecog_ps: int = 0                   # ECOG Performance Status 0–4

    # --- Labs ---
    labs: LabValues = field(default_factory=LabValues)

    # --- Medications ---
    medications: List[str] = field(default_factory=list)

    # --- Metadata ---
    intake_date: date = field(default_factory=date.today)
    clinician_notes: str = ''
    weight_kg: Optional[float] = None
    allergies: List[str] = field(default_factory=list)

    # --- Computed fields (populated by alert logic) ---
    abnormal_labs: List[str] = field(default_factory=list, init=False)
    drug_interactions: List[str] = field(default_factory=list, init=False)
    risk_level: str = field(default='low', init=False)  # 'low', 'moderate', 'high'
    risk_flags: List[str] = field(default_factory=list, init=False)

    def to_flat_dict(self) -> dict:
        """Flatten to a single dict suitable for pandas DataFrame row."""
        return {
            'patient_id': self.patient_id,
            'initials': self.initials,
            'age': self.age,
            'sex': self.sex,
            'diagnosis': self.diagnosis,
            'diagnosis_stage': self.diagnosis_stage,
            'ecog_ps': self.ecog_ps,
            'weight_kg': self.weight_kg,
            # CBC
            'wbc': self.labs.wbc,
            'hemoglobin': self.labs.hemoglobin,
            'platelets': self.labs.platelets,
            'anc': self.labs.anc,
            # Metabolic
            'creatinine': self.labs.creatinine,
            'sodium': self.labs.sodium,
            'potassium': self.labs.potassium,
            'magnesium': self.labs.magnesium,
            'alt': self.labs.alt,
            'ast': self.labs.ast,
            'total_bilirubin': self.labs.total_bilirubin,
            # Medications & alerts
            'medications': '; '.join(self.medications),
            'allergies': '; '.join(self.allergies),
            'abnormal_labs': '; '.join(self.abnormal_labs),
            'drug_interactions': '; '.join(self.drug_interactions),
            'risk_level': self.risk_level,
            'risk_flags': '; '.join(self.risk_flags),
            'intake_date': self.intake_date.isoformat(),
            'clinician_notes': self.clinician_notes,
        }


print('✅ PatientIntake dataclass defined')
print(f'   Fields: demographics, labs (CBC + metabolic), medications, alerts, risk')

✅ PatientIntake dataclass defined
   Fields: demographics, labs (CBC + metabolic), medications, alerts, risk


---
## Section 3 — Clinical Alert Logic

Three functions implement the core clinical decision support:

1. **`flag_abnormal_labs()`** — compares each lab value against published reference ranges and CTCAE Grade thresholds used in oncology
2. **`assess_risk()`** — stratifies patients into low / moderate / high risk based on lab abnormalities, ECOG PS, and clinical context
3. **`check_drug_interactions()`** — screens the medication list against a curated interaction database relevant to oncology

In [20]:
# ─────────────────────────────────────────────
# 3A. Abnormal Lab Flagging
# Reference: CTCAE v5.0 + standard adult reference ranges
# ─────────────────────────────────────────────

def flag_abnormal_labs(patient: PatientIntake) -> List[str]:
    """
    Flag lab values outside normal range or meeting CTCAE Grade ≥2 thresholds.
    Returns list of flag strings describing the abnormality and its severity.

    Grades used (oncology-relevant thresholds):
      ANC: <2.0 = low, <1.5 = Grade 2, <1.0 = Grade 3 (hold chemo), <0.5 = Grade 4
      Plt: <100 = Grade 2, <50 = Grade 3, <25 = Grade 4
      Hgb: <10 = Grade 2, <8 = Grade 3
      Cr:  >1.5x ULN = Grade 2 renal
      Bili: >1.5x ULN = Grade 2 hepatic
      ALT/AST: >3x ULN = Grade 2, >5x ULN = Grade 3
    """
    flags = []
    labs = patient.labs

    # -- WBC --
    if labs.wbc is not None:
        if labs.wbc < 2.0:
            flags.append(f'WBC critically low ({labs.wbc} x10^9/L) — Grade 3/4 leukopenia')
        elif labs.wbc < 4.5:
            flags.append(f'WBC low ({labs.wbc} x10^9/L)')
        elif labs.wbc > 11.0:
            flags.append(f'WBC elevated ({labs.wbc} x10^9/L)')

    # -- ANC (most critical for chemotherapy hold decisions) --
    if labs.anc is not None:
        if labs.anc < 0.5:
            flags.append(f'ANC critically low ({labs.anc} x10^9/L) — Grade 4 neutropenia, HOLD CHEMO')
        elif labs.anc < 1.0:
            flags.append(f'ANC < 1.0 ({labs.anc} x10^9/L) — Grade 3 neutropenia, HOLD CHEMO')
        elif labs.anc < 1.5:
            flags.append(f'ANC < 1.5 ({labs.anc} x10^9/L) — Grade 2 neutropenia, consider delay')
        elif labs.anc < 1.8:
            flags.append(f'ANC borderline low ({labs.anc} x10^9/L)')

    # -- Hemoglobin --
    if labs.hemoglobin is not None:
        lower_limit = 12.0 if patient.sex == 'F' else 13.5
        if labs.hemoglobin < 8.0:
            flags.append(f'Hemoglobin critically low ({labs.hemoglobin} g/dL) — Grade 3 anemia, consider transfusion')
        elif labs.hemoglobin < 10.0:
            flags.append(f'Hemoglobin low ({labs.hemoglobin} g/dL) — Grade 2 anemia')
        elif labs.hemoglobin < lower_limit:
            flags.append(f'Hemoglobin below normal ({labs.hemoglobin} g/dL)')

    # -- Platelets --
    if labs.platelets is not None:
        if labs.platelets < 25:
            flags.append(f'Platelets critically low ({labs.platelets} x10^9/L) — Grade 4 thrombocytopenia, bleeding risk')
        elif labs.platelets < 50:
            flags.append(f'Platelets < 50 ({labs.platelets} x10^9/L) — Grade 3 thrombocytopenia, HOLD CHEMO')
        elif labs.platelets < 100:
            flags.append(f'Platelets < 100 ({labs.platelets} x10^9/L) — Grade 2 thrombocytopenia')

    # -- Creatinine (renal function) --
    if labs.creatinine is not None:
        if labs.creatinine > 3.0:
            flags.append(f'Creatinine critically elevated ({labs.creatinine} mg/dL) — possible renal failure, dose-adjust nephrotoxic agents')
        elif labs.creatinine > 1.95:  # ~1.5x ULN of 1.3
            flags.append(f'Creatinine elevated ({labs.creatinine} mg/dL) — Grade 2 renal toxicity')
        elif labs.creatinine > 1.3:
            flags.append(f'Creatinine above normal ({labs.creatinine} mg/dL)')

    # -- Electrolytes --
    if labs.sodium is not None:
        if labs.sodium < 130:
            flags.append(f'Sodium critically low ({labs.sodium} mEq/L) — Grade 3 hyponatremia')
        elif labs.sodium < 136:
            flags.append(f'Sodium low ({labs.sodium} mEq/L) — hyponatremia')
        elif labs.sodium > 145:
            flags.append(f'Sodium elevated ({labs.sodium} mEq/L) — hypernatremia')

    if labs.potassium is not None:
        if labs.potassium < 3.0:
            flags.append(f'Potassium critically low ({labs.potassium} mEq/L) — Grade 3 hypokalemia, arrhythmia risk')
        elif labs.potassium < 3.5:
            flags.append(f'Potassium low ({labs.potassium} mEq/L) — hypokalemia, replete before chemo')
        elif labs.potassium > 5.5:
            flags.append(f'Potassium elevated ({labs.potassium} mEq/L) — hyperkalemia')

    if labs.magnesium is not None:
        if labs.magnesium < 1.2:
            flags.append(f'Magnesium critically low ({labs.magnesium} mg/dL) — Grade 3 hypomagnesemia')
        elif labs.magnesium < 1.7:
            flags.append(f'Magnesium low ({labs.magnesium} mg/dL) — replete before platinum-based chemo')

    # -- Liver function tests --
    if labs.alt is not None:
        if labs.alt > 280:  # >5x ULN
            flags.append(f'ALT critically elevated ({labs.alt} U/L) — Grade 3 hepatotoxicity, HOLD hepatotoxic agents')
        elif labs.alt > 168:  # >3x ULN
            flags.append(f'ALT elevated ({labs.alt} U/L) — Grade 2 hepatotoxicity')
        elif labs.alt > 56:
            flags.append(f'ALT above normal ({labs.alt} U/L)')

    if labs.ast is not None:
        if labs.ast > 200:  # >5x ULN
            flags.append(f'AST critically elevated ({labs.ast} U/L) — Grade 3 hepatotoxicity')
        elif labs.ast > 120:  # >3x ULN
            flags.append(f'AST elevated ({labs.ast} U/L) — Grade 2 hepatotoxicity')
        elif labs.ast > 40:
            flags.append(f'AST above normal ({labs.ast} U/L)')

    if labs.total_bilirubin is not None:
        if labs.total_bilirubin > 3.0:  # >3x ULN
            flags.append(f'Total bilirubin critically elevated ({labs.total_bilirubin} mg/dL) — Grade 3 hyperbilirubinemia')
        elif labs.total_bilirubin > 1.8:  # >1.5x ULN
            flags.append(f'Total bilirubin elevated ({labs.total_bilirubin} mg/dL) — Grade 2 hyperbilirubinemia')
        elif labs.total_bilirubin > 1.2:
            flags.append(f'Total bilirubin above normal ({labs.total_bilirubin} mg/dL)')

    return flags


print('✅ flag_abnormal_labs() defined')

✅ flag_abnormal_labs() defined


In [21]:
# ─────────────────────────────────────────────
# 3B. Risk Stratification
# ─────────────────────────────────────────────

def assess_risk(patient: PatientIntake) -> Tuple[str, List[str]]:
    """
    Stratify patient into 'low', 'moderate', or 'high' risk based on:
      - Grade 3/4 lab abnormalities (CTCAE)
      - ECOG Performance Status
      - Combination risk factors (electrolyte abnormalities + chemo plan)
      - Chemotherapy hold criteria

    Returns: (risk_level: str, risk_flags: List[str])
    """
    flags = []
    score = 0  # cumulative risk score

    labs = patient.labs

    # ECOG Performance Status
    if patient.ecog_ps >= 3:
        flags.append(f'ECOG PS {patient.ecog_ps} — patient may not tolerate full-dose chemotherapy')
        score += 3
    elif patient.ecog_ps == 2:
        flags.append(f'ECOG PS 2 — reduced performance status, monitor closely')
        score += 1

    # Chemotherapy hold criteria — ANC
    if labs.anc is not None and labs.anc < 1.0:
        flags.append('ANC < 1.0 — standard chemotherapy hold criterion met')
        score += 3

    # Chemotherapy hold criteria — Platelets
    if labs.platelets is not None and labs.platelets < 75:
        flags.append('Platelets < 75 — chemotherapy hold criterion met')
        score += 3

    # Critical anemia
    if labs.hemoglobin is not None and labs.hemoglobin < 8.0:
        flags.append('Hemoglobin < 8 g/dL — consider RBC transfusion before proceeding')
        score += 2

    # Renal impairment affecting drug dosing
    if labs.creatinine is not None and labs.creatinine > 1.95:
        flags.append('Renal impairment — dose-adjust or hold nephrotoxic agents (cisplatin, methotrexate)')
        score += 2

    # Hepatic impairment affecting drug metabolism
    if labs.total_bilirubin is not None and labs.total_bilirubin > 1.8:
        flags.append('Hepatic impairment — dose-adjust hepatically-cleared agents (docetaxel, vincristine)')
        score += 2

    if labs.alt is not None and labs.alt > 280:
        flags.append('Grade 3 hepatotoxicity — evaluate for drug-induced liver injury')
        score += 2

    # Electrolyte abnormalities (relevant for QTc, cardiac risk with certain chemo)
    if labs.potassium is not None and labs.potassium < 3.0:
        flags.append('Severe hypokalemia — QTc prolongation risk, must replete before anthracyclines')
        score += 2

    if labs.magnesium is not None and labs.magnesium < 1.2:
        flags.append('Severe hypomagnesemia — refractory hypokalemia risk, replete before cisplatin')
        score += 1

    # Hyponatremia
    if labs.sodium is not None and labs.sodium < 130:
        flags.append('Severe hyponatremia — evaluate for SIADH, CNS metastases, adrenal insufficiency')
        score += 2

    # Age + ECOG combined
    if patient.age >= 70 and patient.ecog_ps >= 2:
        flags.append('Elderly patient (≥70) with ECOG ≥2 — consider geriatric oncology assessment')
        score += 1

    # Multiple abnormalities
    if len(patient.abnormal_labs) >= 5:
        flags.append(f'Multiple abnormal labs ({len(patient.abnormal_labs)}) — comprehensive review required')
        score += 1

    # Risk stratification thresholds
    if score >= 4:
        risk_level = 'high'
    elif score >= 2:
        risk_level = 'moderate'
    else:
        risk_level = 'low'

    return risk_level, flags


print('✅ assess_risk() defined')

✅ assess_risk() defined


In [22]:
# ─────────────────────────────────────────────
# 3C. Drug Interaction Checking
# ─────────────────────────────────────────────

# Curated interaction database: relevant to outpatient oncology
# Each entry: (drug_a, drug_b, severity, mechanism, recommendation)
DRUG_INTERACTIONS_DB: List[Dict] = [
    {
        'drug_a': 'warfarin', 'drug_b': 'capecitabine',
        'severity': 'MAJOR',
        'mechanism': 'Capecitabine inhibits CYP2C9 → increased warfarin levels → bleeding risk',
        'recommendation': 'Monitor INR closely; consider dose reduction or switch to LMWH'
    },
    {
        'drug_a': 'warfarin', 'drug_b': 'fluorouracil',
        'severity': 'MAJOR',
        'mechanism': '5-FU inhibits CYP2C9 → potentiates warfarin anticoagulation',
        'recommendation': 'Frequent INR monitoring; anticipate 2–3x warfarin dose reduction'
    },
    {
        'drug_a': 'methotrexate', 'drug_b': 'nsaid',
        'severity': 'MAJOR',
        'mechanism': 'NSAIDs reduce renal elimination of methotrexate → toxicity',
        'recommendation': 'Avoid NSAIDs with methotrexate; use acetaminophen for pain'
    },
    {
        'drug_a': 'methotrexate', 'drug_b': 'ibuprofen',
        'severity': 'MAJOR',
        'mechanism': 'NSAID reduces methotrexate renal clearance → methotrexate toxicity',
        'recommendation': 'Avoid combination; hold ibuprofen ≥24h before MTX'
    },
    {
        'drug_a': 'cisplatin', 'drug_b': 'aminoglycoside',
        'severity': 'MAJOR',
        'mechanism': 'Additive nephrotoxicity and ototoxicity',
        'recommendation': 'Avoid concurrent use; if unavoidable, monitor renal function daily'
    },
    {
        'drug_a': 'tamoxifen', 'drug_b': 'fluoxetine',
        'severity': 'MODERATE',
        'mechanism': 'Fluoxetine inhibits CYP2D6 → reduced conversion of tamoxifen to active endoxifen',
        'recommendation': 'Switch to SSRI with lower CYP2D6 inhibition (venlafaxine, citalopram)'
    },
    {
        'drug_a': 'tamoxifen', 'drug_b': 'paroxetine',
        'severity': 'MAJOR',
        'mechanism': 'Paroxetine is potent CYP2D6 inhibitor — substantially reduces tamoxifen efficacy',
        'recommendation': 'Avoid combination; switch to venlafaxine or gabapentin for hot flashes'
    },
    {
        'drug_a': 'imatinib', 'drug_b': 'warfarin',
        'severity': 'MODERATE',
        'mechanism': 'Imatinib inhibits CYP2C9 → elevated warfarin levels',
        'recommendation': 'Use LMWH instead of warfarin when anticoagulation required'
    },
    {
        'drug_a': 'vincristine', 'drug_b': 'azole antifungal',
        'severity': 'MAJOR',
        'mechanism': 'Azoles inhibit CYP3A4 → increased vincristine exposure → neurotoxicity',
        'recommendation': 'Avoid; if antifungal needed, use micafungin or anidulafungin'
    },
    {
        'drug_a': 'vincristine', 'drug_b': 'fluconazole',
        'severity': 'MAJOR',
        'mechanism': 'Fluconazole inhibits CYP3A4 → vincristine accumulation → severe neurotoxicity',
        'recommendation': 'Avoid; switch to echinocandin antifungal'
    },
    {
        'drug_a': 'cyclophosphamide', 'drug_b': 'allopurinol',
        'severity': 'MODERATE',
        'mechanism': 'Allopurinol may inhibit metabolism of cyclophosphamide → increased toxicity',
        'recommendation': 'Monitor CBC closely; consider rasburicase for uric acid management'
    },
    {
        'drug_a': 'doxorubicin', 'drug_b': 'trastuzumab',
        'severity': 'MODERATE',
        'mechanism': 'Additive cardiotoxicity — increased risk of cardiomyopathy and heart failure',
        'recommendation': 'Do not administer concurrently; stagger by ≥24h; monitor LVEF every 3 months'
    },
    {
        'drug_a': 'erlotinib', 'drug_b': 'omeprazole',
        'severity': 'MODERATE',
        'mechanism': 'PPIs raise gastric pH → reduced erlotinib absorption (needs acidic environment)',
        'recommendation': 'Administer erlotinib 2h before or 10h after PPI; consider H2 blocker instead'
    },
    {
        'drug_a': 'ondansetron', 'drug_b': 'aprepitant',
        'severity': 'MINOR',
        'mechanism': 'Minor CYP3A4 interaction; generally well tolerated',
        'recommendation': 'Standard antiemetic combination — no dose adjustment typically required'
    },
    {
        'drug_a': 'prednisone', 'drug_b': 'metformin',
        'severity': 'MODERATE',
        'mechanism': 'Corticosteroids cause hyperglycemia; may cause lactic acidosis risk with metformin if renal function worsens',
        'recommendation': 'Monitor blood glucose closely; may need temporary insulin during high-dose steroid treatment'
    },
]


def check_drug_interactions(patient: PatientIntake) -> List[str]:
    """
    Screen patient's medication list against the oncology drug interaction database.
    Returns list of interaction strings for the patient record.

    Notes on limitations (addressed in Critical Reflection):
      - This is a static, hand-curated subset — not a complete interaction database
      - String matching is naive (does not handle brand names, abbreviations, drug classes)
      - No pharmacokinetic modeling or dose-dependency
    """
    interactions = []
    meds_lower = [m.lower().strip() for m in patient.medications]

    for interaction in DRUG_INTERACTIONS_DB:
        drug_a = interaction['drug_a'].lower()
        drug_b = interaction['drug_b'].lower()

        # Check if both drugs (or substrings matching them) are in the patient's med list
        found_a = any(drug_a in med for med in meds_lower)
        found_b = any(drug_b in med for med in meds_lower)

        if found_a and found_b:
            severity = interaction['severity']
            alert = (
                f"[{severity}] {interaction['drug_a'].title()} + {interaction['drug_b'].title()}: "
                f"{interaction['mechanism']}. → {interaction['recommendation']}"
            )
            interactions.append(alert)

    return interactions


def process_patient(patient: PatientIntake) -> PatientIntake:
    """Run all clinical alert logic on a patient record in place."""
    patient.abnormal_labs = flag_abnormal_labs(patient)
    patient.drug_interactions = check_drug_interactions(patient)
    patient.risk_level, patient.risk_flags = assess_risk(patient)
    return patient


print('✅ check_drug_interactions() and process_patient() defined')
print(f'   Drug interaction database: {len(DRUG_INTERACTIONS_DB)} interaction pairs loaded')

✅ check_drug_interactions() and process_patient() defined
   Drug interaction database: 15 interaction pairs loaded


In [23]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, cwd="/content", capture_output=True, text=True)
    print("CMD:", " ".join(cmd))
    print("OUT:", result.stdout.strip())
    print("ERR:", result.stderr.strip())
    print("─" * 40)
    return result

# First let's see what files exist in /content
import os
files = os.listdir("/content")
print("Files in /content:")
for f in files:
    print(" →", f)

Files in /content:
 → .config
 → Copy_of_KHCC_Patient_Intake_Processing (1).ipynb
 → KHCC_Patient_Intake_Processing.ipynb
 → khcc_intake_output
 → Copy_of_KHCC_Patient_Intake_Processing.ipynb
 → .git
 → sample_data


---
## Section 4 — Patient Dataset (≥15 patients)

The dataset includes 18 realistic oncology patients across common diagnoses seen at KHCC:
- Breast cancer, Non-Hodgkin lymphoma, Colorectal cancer, Lung adenocarcinoma, AML, CML, Ovarian cancer, Cervical cancer, Gastric cancer, Hepatocellular carcinoma

Patients are designed to represent varied clinical profiles: routine pre-chemo review, lab abnormalities requiring holds, multi-drug interactions, high-risk presentations, and elderly patients.

In [ ]:
patients_raw = [

    # ── Patient 1: Routine breast cancer, pre-cycle 3 AC-T ──
    PatientIntake(
        patient_id='KHCC-001', initials='S.A.M.', age=47, sex='F',
        diagnosis='Breast Cancer', diagnosis_stage='Stage II HER2+',
        ecog_ps=1, weight_kg=68.0,
        labs=LabValues(wbc=5.8, hemoglobin=11.8, platelets=210, anc=3.1,
                       creatinine=0.9, sodium=139, potassium=4.1, magnesium=1.9,
                       alt=32, ast=28, total_bilirubin=0.7),
        medications=['doxorubicin', 'cyclophosphamide', 'trastuzumab', 'ondansetron', 'aprepitant'],
        intake_date=date(2025, 6, 2),
        clinician_notes='Cycle 3 of AC-T. Concurrent trastuzumab. Cardiology cleared last cycle.'
    ),

    # ── Patient 2: NHL with Grade 3 neutropenia — chemo hold ──
    PatientIntake(
        patient_id='KHCC-002', initials='M.K.T.', age=55, sex='M',
        diagnosis='Non-Hodgkin Lymphoma', diagnosis_stage='Stage III DLBCL',
        ecog_ps=2, weight_kg=78.5,
        labs=LabValues(wbc=1.8, hemoglobin=9.4, platelets=88, anc=0.7,
                       creatinine=1.1, sodium=138, potassium=3.6, magnesium=1.6,
                       alt=45, ast=38, total_bilirubin=1.0),
        medications=['vincristine', 'fluconazole', 'cyclophosphamide', 'prednisone', 'rituximab'],
        intake_date=date(2025, 6, 2),
        clinician_notes='R-CHOP Cycle 4. On fluconazole for oral candidiasis. Review interaction.'
    ),

    # ── Patient 3: Colorectal cancer on capecitabine + warfarin ──
    PatientIntake(
        patient_id='KHCC-003', initials='R.H.A.', age=63, sex='M',
        diagnosis='Colorectal Cancer', diagnosis_stage='Stage IV metastatic',
        ecog_ps=1, weight_kg=82.0,
        labs=LabValues(wbc=6.2, hemoglobin=12.1, platelets=195, anc=3.8,
                       creatinine=1.0, sodium=140, potassium=4.0, magnesium=1.8,
                       alt=28, ast=31, total_bilirubin=0.9),
        medications=['capecitabine', 'warfarin', 'omeprazole', 'bevacizumab'],
        intake_date=date(2025, 6, 2),
        clinician_notes='XELOX regimen. On warfarin for prior DVT. INR last week 3.2 (elevated).'
    ),

    # ── Patient 4: Lung adenocarcinoma on erlotinib + PPI ──
    PatientIntake(
        patient_id='KHCC-004', initials='L.B.N.', age=58, sex='F',
        diagnosis='Lung Adenocarcinoma', diagnosis_stage='Stage IIIB EGFR-mutant',
        ecog_ps=1, weight_kg=57.0,
        labs=LabValues(wbc=7.1, hemoglobin=11.0, platelets=230, anc=4.2,
                       creatinine=0.8, sodium=137, potassium=3.8, magnesium=2.0,
                       alt=62, ast=55, total_bilirubin=0.8),
        medications=['erlotinib', 'omeprazole', 'ondansetron'],
        intake_date=date(2025, 6, 2),
        clinician_notes='EGFR-TKI therapy. Rash grade 1. Complains of reflux — on PPI.'
    ),

    # ── Patient 5: AML with critical labs ──
    PatientIntake(
        patient_id='KHCC-005', initials='Y.Z.Q.', age=34, sex='M',
        diagnosis='Acute Myeloid Leukemia', diagnosis_stage='Relapsed/Refractory',
        ecog_ps=3, weight_kg=71.0,
        labs=LabValues(wbc=1.2, hemoglobin=7.1, platelets=22, anc=0.3,
                       creatinine=1.6, sodium=132, potassium=2.8, magnesium=1.1,
                       alt=95, ast=88, total_bilirubin=2.1),
        medications=['azacitidine', 'allopurinol', 'fluconazole', 'acyclovir'],
        intake_date=date(2025, 6, 2),
        clinician_notes='Salvage therapy. Multiple transfusions last cycle. Very poor performance status.'
    ),

    # ── Patient 6: CML stable on imatinib + warfarin ──
    PatientIntake(
        patient_id='KHCC-006', initials='A.N.F.', age=42, sex='F',
        diagnosis='Chronic Myeloid Leukemia', diagnosis_stage='Chronic Phase',
        ecog_ps=0, weight_kg=64.0,
        labs=LabValues(wbc=8.5, hemoglobin=13.2, platelets=320, anc=5.1,
                       creatinine=0.7, sodium=141, potassium=4.2, magnesium=2.1,
                       alt=41, ast=35, total_bilirubin=0.6),
        medications=['imatinib', 'warfarin'],
        intake_date=date(2025, 6, 2),
        clinician_notes='PCR response good. On warfarin for mechanical heart valve.'
    ),

    # ── Patient 7: Breast cancer on tamoxifen + paroxetine ──
    PatientIntake(
        patient_id='KHCC-007', initials='H.S.R.', age=51, sex='F',
        diagnosis='Breast Cancer', diagnosis_stage='Stage I ER+/HER2-',
        ecog_ps=0, weight_kg=72.0,
        labs=LabValues(wbc=6.0, hemoglobin=13.5, platelets=260, anc=3.5,
                       creatinine=0.8, sodium=138, potassium=4.0, magnesium=1.9,
                       alt=22, ast=19, total_bilirubin=0.5),
        medications=['tamoxifen', 'paroxetine', 'calcium carbonate', 'vitamin D'],
        intake_date=date(2025, 6, 2),
        clinician_notes='Adjuvant tamoxifen year 2. On paroxetine for hot flashes — discuss CYP2D6 interaction.'
    ),

    # ── Patient 8: Ovarian cancer — methotrexate + ibuprofen ──
    PatientIntake(
        patient_id='KHCC-008', initials='D.M.P.', age=49, sex='F',
        diagnosis='Ovarian Cancer', diagnosis_stage='Stage IIIC high-grade serous',
        ecog_ps=1, weight_kg=61.0,
        labs=LabValues(wbc=5.5, hemoglobin=10.8, platelets=145, anc=3.2,
                       creatinine=1.2, sodium=136, potassium=3.4, magnesium=1.6,
                       alt=30, ast=25, total_bilirubin=0.8),
        medications=['methotrexate', 'ibuprofen', 'ondansetron', 'leucovorin'],
        intake_date=date(2025, 6, 2),
        clinician_notes='MTX-based protocol. Patient taking ibuprofen for joint pain PRN.'
    ),

    # ── Patient 9: Gastric cancer, post-gastrectomy, electrolyte issues ──
    PatientIntake(
        patient_id='KHCC-009', initials='F.W.B.', age=66, sex='M',
        diagnosis='Gastric Cancer', diagnosis_stage='Stage IV',
        ecog_ps=2, weight_kg=55.0,
        labs=LabValues(wbc=4.8, hemoglobin=9.6, platelets=175, anc=2.8,
                       creatinine=1.4, sodium=129, potassium=3.1, magnesium=1.3,
                       alt=48, ast=42, total_bilirubin=1.5),
        medications=['oxaliplatin', 'fluorouracil', 'leucovorin', 'ondansetron'],
        intake_date=date(2025, 6, 2),
        clinician_notes='FOLFOX. Poor oral intake post-gastrectomy. Weight loss 5 kg this month.'
    ),

    # ── Patient 10: Hepatocellular carcinoma with liver dysfunction ──
    PatientIntake(
        patient_id='KHCC-010', initials='K.T.O.', age=59, sex='M',
        diagnosis='Hepatocellular Carcinoma', diagnosis_stage='Unresectable',
        ecog_ps=2, weight_kg=74.0,
        labs=LabValues(wbc=3.9, hemoglobin=10.2, platelets=68, anc=2.1,
                       creatinine=1.1, sodium=133, potassium=3.7, magnesium=1.8,
                       alt=312, ast=278, total_bilirubin=3.8),
        medications=['sorafenib', 'spironolactone', 'furosemide', 'lactulose'],
        intake_date=date(2025, 6, 2),
        clinician_notes='Child-Pugh B. Ascites on diuretics. Sorafenib dose already reduced.'
    ),

    # ── Patient 11: Cervical cancer — standard CCRT ──
    PatientIntake(
        patient_id='KHCC-011', initials='N.E.A.', age=38, sex='F',
        diagnosis='Cervical Cancer', diagnosis_stage='Stage IIB',
        ecog_ps=1, weight_kg=62.0,
        labs=LabValues(wbc=5.3, hemoglobin=11.5, platelets=198, anc=3.0,
                       creatinine=0.9, sodium=140, potassium=4.1, magnesium=2.0,
                       alt=18, ast=22, total_bilirubin=0.6),
        medications=['cisplatin', 'ondansetron', 'dexamethasone'],
        intake_date=date(2025, 6, 3),
        clinician_notes='Weekly cisplatin with concurrent radiotherapy. Week 3 of 5.'
    ),

    # ── Patient 12: Elderly breast cancer, reduced chemo tolerance ──
    PatientIntake(
        patient_id='KHCC-012', initials='W.Q.Z.', age=74, sex='F',
        diagnosis='Breast Cancer', diagnosis_stage='Stage III ER+',
        ecog_ps=2, weight_kg=59.0,
        labs=LabValues(wbc=3.5, hemoglobin=10.0, platelets=135, anc=1.6,
                       creatinine=1.3, sodium=137, potassium=3.6, magnesium=1.7,
                       alt=35, ast=30, total_bilirubin=0.9),
        medications=['paclitaxel', 'trastuzumab', 'metformin', 'lisinopril', 'aspirin'],
        intake_date=date(2025, 6, 3),
        clinician_notes='Weekly paclitaxel. T2DM on metformin. Baseline renal function declining.'
    ),

    # ── Patient 13: NHL, bone marrow suppression ──
    PatientIntake(
        patient_id='KHCC-013', initials='C.B.L.', age=44, sex='M',
        diagnosis='Non-Hodgkin Lymphoma', diagnosis_stage='Stage II Follicular',
        ecog_ps=1, weight_kg=80.0,
        labs=LabValues(wbc=2.9, hemoglobin=9.8, platelets=97, anc=1.4,
                       creatinine=0.9, sodium=139, potassium=3.9, magnesium=1.9,
                       alt=26, ast=23, total_bilirubin=0.7),
        medications=['bendamustine', 'rituximab', 'ondansetron', 'acyclovir'],
        intake_date=date(2025, 6, 3),
        clinician_notes='BR regimen Cycle 3. ANC trending down over last 3 cycles.'
    ),

    # ── Patient 14: Colorectal cancer — well controlled ──
    PatientIntake(
        patient_id='KHCC-014', initials='V.O.D.', age=52, sex='M',
        diagnosis='Colorectal Cancer', diagnosis_stage='Stage II adjuvant',
        ecog_ps=0, weight_kg=84.0,
        labs=LabValues(wbc=6.8, hemoglobin=14.2, platelets=245, anc=4.0,
                       creatinine=1.0, sodium=141, potassium=4.3, magnesium=2.1,
                       alt=25, ast=22, total_bilirubin=0.5),
        medications=['capecitabine', 'folic acid', 'omeprazole'],
        intake_date=date(2025, 6, 3),
        clinician_notes='Adjuvant capecitabine. Tolerating well. No significant complaints.'
    ),

    # ── Patient 15: Lung cancer with cisplatin nephrotoxicity risk ──
    PatientIntake(
        patient_id='KHCC-015', initials='I.R.P.', age=61, sex='M',
        diagnosis='Lung Adenocarcinoma', diagnosis_stage='Stage IIIA',
        ecog_ps=1, weight_kg=76.0,
        labs=LabValues(wbc=5.2, hemoglobin=12.8, platelets=188, anc=3.1,
                       creatinine=1.7, sodium=138, potassium=3.7, magnesium=1.5,
                       alt=30, ast=27, total_bilirubin=0.8),
        medications=['cisplatin', 'pemetrexed', 'dexamethasone', 'gentamicin'],
        intake_date=date(2025, 6, 3),
        clinician_notes='Cisplatin-pemetrexed. On gentamicin for UTI. Creatinine rising.'
    ),

    # ── Patient 16: Breast cancer, routine cycle, no issues ──
    PatientIntake(
        patient_id='KHCC-016', initials='P.J.H.', age=39, sex='F',
        diagnosis='Breast Cancer', diagnosis_stage='Stage I TNBC',
        ecog_ps=0, weight_kg=66.0,
        labs=LabValues(wbc=7.2, hemoglobin=13.8, platelets=278, anc=4.8,
                       creatinine=0.7, sodium=139, potassium=4.0, magnesium=2.0,
                       alt=20, ast=18, total_bilirubin=0.4),
        medications=['doxorubicin', 'cyclophosphamide', 'ondansetron', 'granulocyte-csf'],
        intake_date=date(2025, 6, 3),
        clinician_notes='AC Cycle 2. Tolerating well. G-CSF per protocol.'
    ),

    # ── Patient 17: AML — critical hepatic + cytopenias ──
    PatientIntake(
        patient_id='KHCC-017', initials='B.U.K.', age=29, sex='M',
        diagnosis='Acute Myeloid Leukemia', diagnosis_stage='Newly Diagnosed',
        ecog_ps=2, weight_kg=68.0,
        labs=LabValues(wbc=0.8, hemoglobin=7.8, platelets=14, anc=0.2,
                       creatinine=0.9, sodium=136, potassium=4.4, magnesium=1.9,
                       alt=85, ast=72, total_bilirubin=2.8),
        medications=['cytarabine', 'daunorubicin', 'allopurinol', 'acyclovir', 'micafungin'],
        intake_date=date(2025, 6, 4),
        clinician_notes='7+3 induction. Profound cytopenias expected. Monitor hepatic function daily.'
    ),

    # ── Patient 18: Elderly NHL with polypharmacy ──
    PatientIntake(
        patient_id='KHCC-018', initials='G.L.T.', age=72, sex='F',
        diagnosis='Non-Hodgkin Lymphoma', diagnosis_stage='Stage IV DLBCL',
        ecog_ps=2, weight_kg=53.0,
        labs=LabValues(wbc=4.1, hemoglobin=9.2, platelets=112, anc=2.3,
                       creatinine=1.3, sodium=134, potassium=3.2, magnesium=1.4,
                       alt=52, ast=44, total_bilirubin=1.1),
        medications=['rituximab', 'cyclophosphamide', 'vincristine', 'prednisone',
                     'fluconazole', 'metformin', 'lisinopril'],
        intake_date=date(2025, 6, 4),
        clinician_notes='R-CVP. Diabetic. On fluconazole for thrush. Multiple comorbidities.'
    ),
]

# Process all patients through alert logic
patients = [process_patient(p) for p in patients_raw]

print(f'✅ {len(patients)} patients processed')
print(f'   Date range: {min(p.intake_date for p in patients)} to {max(p.intake_date for p in patients)}')
print(f'   Diagnoses: {len(set(p.diagnosis for p in patients))} unique')

✅ 18 patients processed
   Date range: 2025-06-02 to 2025-06-04
   Diagnoses: 10 unique


In [ ]:
# Build master DataFrame
df_all = pd.DataFrame([p.to_flat_dict() for p in patients])

print('Master DataFrame shape:', df_all.shape)
print()
print('Risk level distribution:')
print(df_all['risk_level'].value_counts())
print()
print('Patients with drug interactions:', df_all[df_all['drug_interactions'] != ''].shape[0])
print()
df_all[['patient_id', 'initials', 'age', 'diagnosis', 'risk_level', 'ecog_ps']].head(10)

Master DataFrame shape: (18, 27)

Risk level distribution:
risk_level
low         11
high         5
moderate     2
Name: count, dtype: int64

Patients with drug interactions: 8



,patient_id,initials,age,diagnosis,risk_level,ecog_ps
0,KHCC-001,S.A.M.,47,Breast Cancer,low,1
1,KHCC-002,M.K.T.,55,Non-Hodgkin Lymphoma,high,2
2,KHCC-003,R.H.A.,63,Colorectal Cancer,low,1
3,KHCC-004,L.B.N.,58,Lung Adenocarcinoma,low,1
4,KHCC-005,Y.Z.Q.,34,Acute Myeloid Leukemia,high,3
5,KHCC-006,A.N.F.,42,Chronic Myeloid Leukemia,low,0
6,KHCC-007,H.S.R.,51,Breast Cancer,low,0
7,KHCC-008,D.M.P.,49,Ovarian Cancer,low,1
8,KHCC-009,F.W.B.,66,Gastric Cancer,high,2
9,KHCC-010,K.T.O.,59,Hepatocellular Carcinoma,high,2


---
## Section 5 — Group-Level Statistical Analysis by Diagnosis

In [ ]:
# ─────────────────────────────────────────────
# 5A. Summary statistics grouped by diagnosis
# ─────────────────────────────────────────────

numeric_labs = ['wbc', 'hemoglobin', 'platelets', 'anc', 'creatinine', 'alt', 'ast']

print('=' * 65)
print('GROUP-LEVEL STATISTICS BY DIAGNOSIS')
print('=' * 65)

stats_by_diagnosis = df_all.groupby('diagnosis')[numeric_labs].agg(['mean', 'min', 'max']).round(2)
print(stats_by_diagnosis.to_string())
print()

GROUP-LEVEL STATISTICS BY DIAGNOSIS
                           wbc           hemoglobin             platelets             anc           creatinine               alt               ast          
                          mean  min  max       mean   min   max      mean  min  max  mean  min  max       mean  min  max    mean  min  max    mean  min  max
diagnosis                                                                                                                                                   
Acute Myeloid Leukemia    1.00  0.8  1.2       7.45   7.1   7.8     18.00   14   22  0.25  0.2  0.3       1.25  0.9  1.6   90.00   85   95   80.00   72   88
Breast Cancer             5.62  3.5  7.2      12.28  10.0  13.8    220.75  135  278  3.25  1.6  4.8       0.92  0.7  1.3   27.25   20   35   23.75   18   30
Cervical Cancer           5.30  5.3  5.3      11.50  11.5  11.5    198.00  198  198  3.00  3.0  3.0       0.90  0.9  0.9   18.00   18   18   22.00   22   22
Chronic Myeloid Leukem

In [ ]:
# ─────────────────────────────────────────────
# 5B. Risk distribution per diagnosis
# ─────────────────────────────────────────────

print('=' * 55)
print('RISK LEVEL BY DIAGNOSIS')
print('=' * 55)

risk_pivot = pd.crosstab(df_all['diagnosis'], df_all['risk_level'],
                          margins=True, margins_name='Total')
# Reorder columns
cols_order = [c for c in ['low', 'moderate', 'high', 'Total'] if c in risk_pivot.columns]
print(risk_pivot[cols_order].to_string())
print()

RISK LEVEL BY DIAGNOSIS
risk_level                low  moderate  high  Total
diagnosis                                           
Acute Myeloid Leukemia      0         0     2      2
Breast Cancer               3         1     0      4
Cervical Cancer             1         0     0      1
Chronic Myeloid Leukemia    1         0     0      1
Colorectal Cancer           2         0     0      2
Gastric Cancer              0         0     1      1
Hepatocellular Carcinoma    0         0     1      1
Lung Adenocarcinoma         2         0     0      2
Non-Hodgkin Lymphoma        1         1     1      3
Ovarian Cancer              1         0     0      1
Total                      11         2     5     18



In [ ]:
# ─────────────────────────────────────────────
# 5C. Aggregate alerts summary
# ─────────────────────────────────────────────

print('=' * 55)
print('DAILY INTAKE SUMMARY')
print('=' * 55)

total = len(patients)
high_risk = df_all[df_all['risk_level'] == 'high'].shape[0]
mod_risk = df_all[df_all['risk_level'] == 'moderate'].shape[0]
low_risk = df_all[df_all['risk_level'] == 'low'].shape[0]
with_interactions = df_all[df_all['drug_interactions'] != ''].shape[0]
chemo_hold = df_all[df_all['abnormal_labs'].str.contains('HOLD CHEMO', na=False)].shape[0]

print(f'Total patients reviewed:       {total}')
print(f'  High risk:                   {high_risk} ({100*high_risk/total:.0f}%)')
print(f'  Moderate risk:               {mod_risk} ({100*mod_risk/total:.0f}%)')
print(f'  Low risk:                    {low_risk} ({100*low_risk/total:.0f}%)')
print(f'Drug interaction alerts:       {with_interactions}')
print(f'Chemo hold criteria met:       {chemo_hold}')
print(f'Patients with abnormal labs:   {df_all[df_all["abnormal_labs"] != ""].shape[0]}')
print()

DAILY INTAKE SUMMARY
Total patients reviewed:       18
  High risk:                   5 (28%)
  Moderate risk:               2 (11%)
  Low risk:                    11 (61%)
Drug interaction alerts:       8
Chemo hold criteria met:       3
Patients with abnormal labs:   14



In [ ]:
# ─────────────────────────────────────────────
# 5D. Mean lab values by risk stratum
# ─────────────────────────────────────────────

print('MEAN LAB VALUES BY RISK STRATUM')
print('-' * 55)
risk_labs = df_all.groupby('risk_level')[['anc', 'hemoglobin', 'platelets', 'creatinine', 'alt']].mean().round(2)
print(risk_labs.to_string())
print()
print('(Note: ANC, Hgb, and Plt are lowest in high-risk stratum as expected)')

MEAN LAB VALUES BY RISK STRATUM
-------------------------------------------------------
             anc  hemoglobin  platelets  creatinine     alt
risk_level                                                 
high        1.22        8.82      73.40        1.22  117.00
low         3.56       12.23     215.09        0.96   30.36
moderate    1.95        9.60     123.50        1.30   43.50

(Note: ANC, Hgb, and Plt are lowest in high-risk stratum as expected)


---
## Section 6 — CSV Export (3 Files)

In [ ]:
# ─────────────────────────────────────────────
# File 1: All patients
# ─────────────────────────────────────────────

path_all = os.path.join(OUTPUT_DIR, 'all_patients.csv')
df_all.to_csv(path_all, index=False)
print(f'✅ File 1 exported: {path_all}')
print(f'   Rows: {len(df_all)}  |  Columns: {len(df_all.columns)}')
print()

# ─────────────────────────────────────────────
# File 2: High-risk patients only
# ─────────────────────────────────────────────

df_high_risk = df_all[df_all['risk_level'] == 'high'].copy()
# Add a human-readable alert summary column
df_high_risk['alert_summary'] = df_high_risk.apply(
    lambda row: ' | '.join(filter(None, [
        f"LABS: {row['abnormal_labs'][:80]}..." if len(str(row['abnormal_labs'])) > 80 else f"LABS: {row['abnormal_labs']}",
        f"RISK: {row['risk_flags'][:80]}" if row['risk_flags'] else ''
    ])), axis=1
)

path_high_risk = os.path.join(OUTPUT_DIR, 'high_risk_patients.csv')
df_high_risk.to_csv(path_high_risk, index=False)
print(f'✅ File 2 exported: {path_high_risk}')
print(f'   Rows: {len(df_high_risk)}  |  High-risk patients:')
for _, row in df_high_risk.iterrows():
    print(f'   • {row["patient_id"]} ({row["initials"]}) — {row["diagnosis"]} — ECOG {row["ecog_ps"]}')
print()

# ─────────────────────────────────────────────
# File 3: Drug interaction alerts
# ─────────────────────────────────────────────

df_interactions_raw = df_all[df_all['drug_interactions'] != ''].copy()

# Explode to one row per interaction alert
df_interactions_exploded = []
for _, row in df_interactions_raw.iterrows():
    alerts = [a.strip() for a in row['drug_interactions'].split('\n') if a.strip()]
    # If semicolons were used as separator
    if len(alerts) == 1:
        alerts = [a.strip() for a in row['drug_interactions'].split(';') if a.strip()]
    for alert in alerts:
        df_interactions_exploded.append({
            'patient_id': row['patient_id'],
            'initials': row['initials'],
            'diagnosis': row['diagnosis'],
            'ecog_ps': row['ecog_ps'],
            'risk_level': row['risk_level'],
            'medications': row['medications'],
            'interaction_alert': alert,
            'severity': 'MAJOR' if '[MAJOR]' in alert else ('MODERATE' if '[MODERATE]' in alert else 'MINOR'),
            'intake_date': row['intake_date'],
        })

df_interactions = pd.DataFrame(df_interactions_exploded)

path_interactions = os.path.join(OUTPUT_DIR, 'drug_interaction_alerts.csv')
df_interactions.to_csv(path_interactions, index=False)
print(f'✅ File 3 exported: {path_interactions}')
print(f'   Rows: {len(df_interactions)}  |  Unique patients with alerts: {df_interactions["patient_id"].nunique()}')
print(f'   MAJOR interactions: {(df_interactions["severity"] == "MAJOR").sum()}')
print(f'   MODERATE interactions: {(df_interactions["severity"] == "MODERATE").sum()}')
print()
print('─' * 55)
print(f'All 3 CSV files written to: ./{OUTPUT_DIR}/')

✅ File 1 exported: khcc_intake_output/all_patients.csv
   Rows: 18  |  Columns: 27

✅ File 2 exported: khcc_intake_output/high_risk_patients.csv
   Rows: 5  |  High-risk patients:
   • KHCC-002 (M.K.T.) — Non-Hodgkin Lymphoma — ECOG 2
   • KHCC-005 (Y.Z.Q.) — Acute Myeloid Leukemia — ECOG 3
   • KHCC-009 (F.W.B.) — Gastric Cancer — ECOG 2
   • KHCC-010 (K.T.O.) — Hepatocellular Carcinoma — ECOG 2
   • KHCC-017 (B.U.K.) — Acute Myeloid Leukemia — ECOG 2

✅ File 3 exported: khcc_intake_output/drug_interaction_alerts.csv
   Rows: 21  |  Unique patients with alerts: 8
   MAJOR interactions: 5
   MODERATE interactions: 4

───────────────────────────────────────────────────────
All 3 CSV files written to: ./khcc_intake_output/


---
## Section 7 — Simulated Email Summary

In [ ]:
def generate_email_summary(patients: List[PatientIntake], run_date: date = None) -> str:
    """
    Generate a simulated daily intake email summary in the style of
    an internal clinical communication at KHCC outpatient oncology.
    """
    if run_date is None:
        run_date = date.today()

    high_risk = [p for p in patients if p.risk_level == 'high']
    moderate_risk = [p for p in patients if p.risk_level == 'moderate']
    with_interactions = [p for p in patients if p.drug_interactions]
    chemo_hold = [p for p in patients if any('HOLD CHEMO' in f for f in p.abnormal_labs)]

    lines = [
        '=' * 70,
        'TO:      Outpatient Oncology Team, KHCC',
        'FROM:    Automated Intake Processing System v1.0',
        f'DATE:    {run_date.strftime("%A, %d %B %Y")}',
        'SUBJECT: Daily Patient Intake Report — Automated Clinical Alerts',
        '=' * 70,
        '',
        '── SUMMARY ──',
        f'  Total patients processed:     {len(patients)}',
        f'  High-risk patients:           {len(high_risk)}',
        f'  Moderate-risk patients:       {len(moderate_risk)}',
        f'  Drug interaction alerts:      {len(with_interactions)} patients ({sum(len(p.drug_interactions) for p in patients)} interactions)',
        f'  Chemotherapy hold criteria:   {len(chemo_hold)} patients',
        '',
        '── HIGH-RISK PATIENTS REQUIRING IMMEDIATE REVIEW ──',
    ]

    if high_risk:
        for p in high_risk:
            lines.append(f'')
            lines.append(f'  Patient: {p.patient_id} ({p.initials}) | Age {p.age} {p.sex} | {p.diagnosis} {p.diagnosis_stage or ""}')
            lines.append(f'  ECOG PS: {p.ecog_ps} | Risk Score: HIGH')
            for flag in p.risk_flags[:3]:  # Top 3 flags
                lines.append(f'    ⚠ {flag}')
            if p.drug_interactions:
                lines.append(f'  Drug interactions ({len(p.drug_interactions)}):')
                for ix in p.drug_interactions[:2]:
                    lines.append(f'    ✗ {ix[:100]}...')
    else:
        lines.append('  No high-risk patients today.')

    lines += [
        '',
        '── CHEMOTHERAPY HOLD CRITERIA MET ──',
    ]

    if chemo_hold:
        for p in chemo_hold:
            hold_reasons = [f for f in p.abnormal_labs if 'HOLD CHEMO' in f]
            lines.append(f'  {p.patient_id} ({p.initials}) — {p.diagnosis}: {hold_reasons[0]}')
    else:
        lines.append('  No patients meeting chemotherapy hold criteria today.')

    lines += [
        '',
        '── DRUG INTERACTION ALERTS ──',
    ]

    for p in with_interactions:
        lines.append(f'  {p.patient_id} ({p.initials}) — {len(p.drug_interactions)} alert(s):')
        for ix in p.drug_interactions:
            severity = 'MAJOR' if '[MAJOR]' in ix else 'MODERATE'
            lines.append(f'    [{severity}] {ix[:110]}')

    lines += [
        '',
        '── ATTACHMENTS ──',
        '  1. all_patients.csv',
        '  2. high_risk_patients.csv',
        '  3. drug_interaction_alerts.csv',
        '',
        '── DISCLAIMER ──',
        '  This report is generated by an automated rule-based system.',
        '  All alerts must be reviewed by a qualified clinician before action.',
        '  This system does not replace clinical judgment.',
        '  Contact: clinical-informatics@khcc.jo for system issues.',
        '',
        '=' * 70,
    ]

    return '\n'.join(lines)


email_text = generate_email_summary(patients, run_date=date(2025, 6, 4))
print(email_text)

# Save email to file
with open(os.path.join(OUTPUT_DIR, 'daily_intake_email.txt'), 'w') as f:
    f.write(email_text)
print(f'\n✅ Email summary saved to {OUTPUT_DIR}/daily_intake_email.txt')

TO:      Outpatient Oncology Team, KHCC
FROM:    Automated Intake Processing System v1.0
DATE:    Wednesday, 04 June 2025
SUBJECT: Daily Patient Intake Report — Automated Clinical Alerts

── SUMMARY ──
  Total patients processed:     18
  High-risk patients:           5
  Moderate-risk patients:       2
  Drug interaction alerts:      8 patients (10 interactions)
  Chemotherapy hold criteria:   3 patients

── HIGH-RISK PATIENTS REQUIRING IMMEDIATE REVIEW ──

  Patient: KHCC-002 (M.K.T.) | Age 55 M | Non-Hodgkin Lymphoma Stage III DLBCL
  ECOG PS: 2 | Risk Score: HIGH
    ⚠ ECOG PS 2 — reduced performance status, monitor closely
    ⚠ ANC < 1.0 — standard chemotherapy hold criterion met
    ⚠ Multiple abnormal labs (5) — comprehensive review required
  Drug interactions (1):
    ✗ [MAJOR] Vincristine + Fluconazole: Fluconazole inhibits CYP3A4 → vincristine accumulation → severe n...

  Patient: KHCC-005 (Y.Z.Q.) | Age 34 M | Acute Myeloid Leukemia Relapsed/Refractory
  ECOG PS: 3 | Risk

In [ ]:
git_commit("feat: add 18-patient dataset, group stats, CSV export, and email summary")


❌ Error: 


---
## Section 8 — Critical Reflection

> **Word count target: 200–400 words. Addresses four required areas.**

### Critical Reflection: Failure Modes, Missing Checks, PHI, and Testing

---

**Failure Modes in Alert Logic**

The most significant failure mode in this system is its reliance on **exact string matching** for both drug identification and lab thresholds. A patient listed as taking "Xeloda" (the brand name for capecitabine) will not trigger the warfarin–capecitabine interaction, even though the clinical risk is identical. Similarly, the system applies **population-level reference ranges** without accounting for patient-specific baselines: a patient with chronic kidney disease may have a normal creatinine of 1.9 mg/dL, which our logic would flag as Grade 2 toxicity when it is actually their stable baseline. This creates **false positive fatigue** — a known cause of alert dismissal in clinical decision support systems, documented extensively in the CDS literature.

The risk scoring system is additive and unweighted, meaning that a patient with five minor abnormalities scores higher than one with a single critical ANC of 0.1. A prioritization model based on **decision tree logic or weighted clinical rules** would better reflect actual clinical urgency.

**Missing Clinical Checks**

Several critical checks are absent from this prototype: (1) **QTc interval monitoring** — multiple common antiemetics (ondansetron) and antifungals (fluconazole) prolong QTc, and risk compounds when combined; (2) **body surface area (BSA) / renal function–based dose calculation** — the system flags abnormal creatinine but does not calculate creatinine clearance (CrCl) or suggest dose adjustments; (3) **allergy cross-reactivity** — the allergy field is stored but not cross-referenced with the medication list; and (4) **trending over multiple cycles** — a single lab value is less clinically meaningful than a downward trend (see Stretch Challenge Section 9).

**PHI and Security Considerations**

In this demonstration, patient identifiers have been reduced to initials. A real deployment would require: (1) **role-based access control (RBAC)** with audit logging compliant with Jordan's Health Data Protection regulations and HIPAA-equivalent standards; (2) **data encryption at rest and in transit** (AES-256, TLS 1.3); (3) **de-identification pipelines** before any data is exported to shared storage; and (4) **strict data retention policies** — exported CSVs must not persist in unsecured cloud storage.

**Testing Strategy**

A robust testing strategy would include: (1) **unit tests** for each alert function using known edge cases (ANC = 0.5 exactly, Cr = 1.95); (2) **integration tests** with synthetic patient datasets covering all possible abnormality combinations; (3) **clinical validation** — retrospective review of 3–6 months of actual KHCC intake records to measure sensitivity/specificity of the alert logic against clinician-verified outcomes; and (4) **regression tests** to ensure new rules do not inadvertently suppress existing alerts.

---
*Reflection word count: ~360 words*

---
## Section 9 — Stretch Challenge

Three stretch deliverables are implemented:
1. **Email Report Generator** — structured HTML-style email with severity tiers
2. **Multi-Day Trend Analysis** — simulated CBC trend data over 3 cycles to detect deteriorating counts
3. **Deployment Proposal** — structured plan for real-world integration at KHCC

In [ ]:
# ─────────────────────────────────────────────
# Stretch 1: Enhanced Email Report Generator
# ─────────────────────────────────────────────

def generate_html_email_report(patients: List[PatientIntake], run_date: date = None) -> str:
    """
    Generate an HTML-formatted email report with color-coded severity tiers.
    Suitable for rendering in email clients (Outlook, Gmail).
    """
    if run_date is None:
        run_date = date.today()

    high_risk = [p for p in patients if p.risk_level == 'high']
    moderate_risk = [p for p in patients if p.risk_level == 'moderate']
    chemo_hold = [p for p in patients if any('HOLD CHEMO' in f for f in p.abnormal_labs)]
    major_interactions = [p for p in patients if any('[MAJOR]' in ix for ix in p.drug_interactions)]

    def patient_row(p: PatientIntake, color: str) -> str:
        flags_html = ''.join(f'<li>{f}</li>' for f in p.risk_flags[:3])
        ix_html = ''.join(f'<li style="color:#c0392b">{ix[:120]}</li>' for ix in p.drug_interactions[:2])
        return f"""
        <tr>
          <td style="padding:8px; border:1px solid #ddd; background:{color}; font-weight:bold">{p.patient_id}</td>
          <td style="padding:8px; border:1px solid #ddd">{p.initials}, {p.age}{p.sex}</td>
          <td style="padding:8px; border:1px solid #ddd">{p.diagnosis}</td>
          <td style="padding:8px; border:1px solid #ddd">ECOG {p.ecog_ps}</td>
          <td style="padding:8px; border:1px solid #ddd">
            ANC {p.labs.anc} | Hgb {p.labs.hemoglobin} | Plt {p.labs.platelets}
          </td>
          <td style="padding:8px; border:1px solid #ddd">
            <ul style="margin:0; padding-left:16px">{flags_html}</ul>
            {('<ul style="margin:4px 0 0; padding-left:16px">' + ix_html + '</ul>') if ix_html else ''}
          </td>
        </tr>"""

    high_rows = ''.join(patient_row(p, '#ffeaea') for p in high_risk)
    mod_rows = ''.join(patient_row(p, '#fff8e1') for p in moderate_risk)

    html = f"""
<!DOCTYPE html>
<html>
<head><meta charset="utf-8"><title>KHCC Daily Intake Report</title></head>
<body style="font-family: Arial, sans-serif; font-size: 13px; color: #222; max-width: 900px; margin: 0 auto; padding: 20px">

<div style="background:#003366; color:white; padding:16px 20px; border-radius:4px">
  <h2 style="margin:0">KHCC Outpatient Oncology — Daily Intake Report</h2>
  <p style="margin:4px 0 0; font-size:12px">{run_date.strftime('%A, %d %B %Y')} &nbsp;|&nbsp; Automated Clinical Alerts System</p>
</div>

<table style="width:100%; border-collapse:collapse; margin-top:16px">
  <tr>
    <td style="background:#c0392b; color:white; padding:10px 16px; border-radius:4px; text-align:center; width:24%">
      <div style="font-size:24px; font-weight:bold">{len(high_risk)}</div>
      <div>HIGH RISK</div>
    </td>
    <td style="width:2%"></td>
    <td style="background:#e67e22; color:white; padding:10px 16px; border-radius:4px; text-align:center; width:24%">
      <div style="font-size:24px; font-weight:bold">{len(moderate_risk)}</div>
      <div>MODERATE RISK</div>
    </td>
    <td style="width:2%"></td>
    <td style="background:#c0392b; color:white; padding:10px 16px; border-radius:4px; text-align:center; width:24%">
      <div style="font-size:24px; font-weight:bold">{len(chemo_hold)}</div>
      <div>CHEMO HOLD</div>
    </td>
    <td style="width:2%"></td>
    <td style="background:#8e44ad; color:white; padding:10px 16px; border-radius:4px; text-align:center; width:24%">
      <div style="font-size:24px; font-weight:bold">{len(major_interactions)}</div>
      <div>MAJOR INTERACTIONS</div>
    </td>
  </tr>
</table>

<h3 style="color:#c0392b; margin-top:24px">High-Risk Patients</h3>
<table style="width:100%; border-collapse:collapse">
  <tr style="background:#003366; color:white">
    <th style="padding:8px; text-align:left">ID</th>
    <th style="padding:8px; text-align:left">Patient</th>
    <th style="padding:8px; text-align:left">Diagnosis</th>
    <th style="padding:8px; text-align:left">ECOG</th>
    <th style="padding:8px; text-align:left">Key Labs</th>
    <th style="padding:8px; text-align:left">Alerts</th>
  </tr>
  {high_rows}
</table>

<h3 style="color:#e67e22; margin-top:20px">Moderate-Risk Patients</h3>
<table style="width:100%; border-collapse:collapse">
  <tr style="background:#003366; color:white">
    <th style="padding:8px; text-align:left">ID</th>
    <th style="padding:8px; text-align:left">Patient</th>
    <th style="padding:8px; text-align:left">Diagnosis</th>
    <th style="padding:8px; text-align:left">ECOG</th>
    <th style="padding:8px; text-align:left">Key Labs</th>
    <th style="padding:8px; text-align:left">Alerts</th>
  </tr>
  {mod_rows}
</table>

<p style="margin-top:24px; padding:12px; background:#f0f0f0; border-left:4px solid #003366; font-size:12px">
  <strong>Disclaimer:</strong> This report is generated by an automated rule-based system.
  All alerts must be verified by a qualified clinician before any clinical action is taken.
  This system is a decision-support aid, not a replacement for clinical judgment.
</p>

</body></html>"""
    return html


html_report = generate_html_email_report(patients, run_date=date(2025, 6, 4))

html_path = os.path.join(OUTPUT_DIR, 'daily_intake_report.html')
with open(html_path, 'w') as f:
    f.write(html_report)

print(f'✅ HTML email report saved: {html_path}')
print(f'   Open this file in a browser to preview the formatted report.')

✅ HTML email report saved: khcc_intake_output/daily_intake_report.html
   Open this file in a browser to preview the formatted report.


In [ ]:
# ─────────────────────────────────────────────
# Stretch 2: Multi-Day Trend Analysis
# Simulates CBC trends across 3 treatment cycles
# and flags patients with deteriorating trajectories.
# ─────────────────────────────────────────────

def simulate_multi_cycle_trend(
    patient_id: str,
    baseline_anc: float,
    baseline_hgb: float,
    baseline_plt: float,
    cycles: int = 4,
    anc_drift: float = -0.3,  # per cycle
    hgb_drift: float = -0.5,
    plt_drift: float = -15.0,
    noise: float = 0.1
) -> pd.DataFrame:
    """
    Simulate lab trends over multiple treatment cycles with physiologically
    realistic drift (cumulative bone marrow suppression) and random noise.
    """
    np.random.seed(42)
    rows = []
    for cycle in range(1, cycles + 1):
        # Day 1 of cycle (pre-treatment nadir recovery)
        recovery_factor = 1.0 + (0.2 * np.random.randn() * noise)
        anc = max(0.1, baseline_anc + anc_drift * (cycle - 1) * recovery_factor + np.random.randn() * 0.2)
        hgb = max(6.0, baseline_hgb + hgb_drift * (cycle - 1) + np.random.randn() * 0.3)
        plt = max(10, baseline_plt + plt_drift * (cycle - 1) + np.random.randn() * 10)

        rows.append({
            'patient_id': patient_id,
            'cycle': cycle,
            'day': 1,
            'anc': round(anc, 2),
            'hemoglobin': round(hgb, 2),
            'platelets': round(plt, 0),
            'anc_flag': 'HOLD' if anc < 1.0 else ('WATCH' if anc < 1.5 else 'OK'),
            'hgb_flag': 'CRITICAL' if hgb < 8 else ('LOW' if hgb < 10 else 'OK'),
            'plt_flag': 'HOLD' if plt < 75 else ('LOW' if plt < 100 else 'OK'),
        })
    return pd.DataFrame(rows)


def flag_deteriorating_trend(trend_df: pd.DataFrame, column: str, threshold: float) -> bool:
    """
    Returns True if the column shows a consistent downward trend across cycles
    AND the last value is below the threshold.
    """
    values = trend_df[column].tolist()
    is_declining = all(values[i] >= values[i+1] for i in range(len(values)-1))
    return is_declining and values[-1] < threshold


# Simulate trends for 4 patients with varying profiles
trend_scenarios = [
    ('KHCC-002', 3.5, 12.0, 180, -0.5, -0.8, -25),   # Gradual deterioration
    ('KHCC-013', 3.0, 11.5, 155, -0.4, -0.6, -20),   # Moderate decline
    ('KHCC-016', 5.2, 14.0, 290, -0.1, -0.2, -5),    # Stable, minimal decline
    ('KHCC-005', 1.5, 9.5, 80,  -0.4, -0.7, -18),    # Already low, critical trajectory
]

trend_frames = []
for pid, anc, hgb, plt, ad, hd, pd_ in trend_scenarios:
    df_trend = simulate_multi_cycle_trend(pid, anc, hgb, plt, cycles=4,
                                           anc_drift=ad, hgb_drift=hd, plt_drift=pd_)
    trend_frames.append(df_trend)

df_trends = pd.concat(trend_frames, ignore_index=True)

print('MULTI-CYCLE CBC TREND TABLE')
print('=' * 60)
print(df_trends.to_string(index=False))
print()

# Trend deterioration detection
print('DETERIORATING TREND DETECTION')
print('-' * 60)
for pid in df_trends['patient_id'].unique():
    subset = df_trends[df_trends['patient_id'] == pid].sort_values('cycle')
    anc_alert = flag_deteriorating_trend(subset, 'anc', threshold=1.5)
    hgb_alert = flag_deteriorating_trend(subset, 'hemoglobin', threshold=10.0)
    plt_alert = flag_deteriorating_trend(subset, 'platelets', threshold=100)

    alerts = []
    if anc_alert: alerts.append('ANC declining trend below 1.5')
    if hgb_alert: alerts.append('Hgb declining trend below 10')
    if plt_alert: alerts.append('Platelets declining trend below 100')

    if alerts:
        print(f'  {pid}: ⚠ {" | ".join(alerts)}')
    else:
        print(f'  {pid}: ✓ No significant declining trend detected')

# Export trend data
trend_path = os.path.join(OUTPUT_DIR, 'cbc_trend_analysis.csv')
df_trends.to_csv(trend_path, index=False)
print(f'\n✅ Trend data exported: {trend_path}')

MULTI-CYCLE CBC TREND TABLE
patient_id  cycle  day  anc  hemoglobin  platelets anc_flag hgb_flag plt_flag
  KHCC-002      1    1 3.47       12.19      195.0       OK       OK       OK
  KHCC-002      2    1 2.96       11.67      163.0       OK       OK       OK
  KHCC-002      3    1 2.62       10.26      125.0       OK       OK       OK
  KHCC-002      4    1 1.61        9.08       99.0       OK      LOW      LOW
  KHCC-013      1    1 2.97       11.69      170.0       OK       OK       OK
  KHCC-013      2    1 2.56       11.37      143.0       OK       OK       OK
  KHCC-013      3    1 2.32       10.16      110.0       OK       OK       OK
  KHCC-013      4    1 1.41        9.18       89.0    WATCH      LOW      LOW
  KHCC-016      1    1 5.17       14.19      305.0       OK       OK       OK
  KHCC-016      2    1 5.05       14.27      293.0       OK       OK       OK
  KHCC-016      3    1 5.11       13.46      275.0       OK       OK       OK
  KHCC-016      4    1 4.52       12

### Stretch 3 — Deployment Proposal: KHCC Outpatient Oncology Intake Automation

---

#### Executive Summary

This proposal outlines a phased deployment of the automated patient intake processing system at KHCC's outpatient oncology clinic, replacing a 2–3 hour daily manual workflow performed by data entry clerks.

---

#### Phase 1 — Pilot (Months 1–3): Shadow Mode

| Item | Detail |
|------|---------|
| Scope | 2 oncology sub-specialty clinics (breast, lymphoma) |
| Mode | System runs in parallel with manual review; clinician compares outputs |
| Goal | Measure sensitivity/specificity of alert logic vs. clinician ground truth |
| Data | De-identified records only; no PHI transmitted outside HIS |
| Success criterion | ≥90% sensitivity for Grade 3/4 events, false positive rate ≤20% |

#### Phase 2 — Supervised Deployment (Months 4–9)

- System generates daily report automatically at 06:30 AM before morning rounds
- Oncology pharmacist reviews all drug interaction alerts before clinician briefing
- Charge nurse reviews all high-risk and chemo-hold flags
- Feedback mechanism: every dismissed alert is logged with reason (supports ML training)

#### Phase 3 — Full Integration (Months 10+)

- Integration with KHCC's Hospital Information System (HIS) via HL7 FHIR API
- Direct push of alerts to clinician mobile app (read-only, acknowledgment required)
- Trend analysis module activated for patients with ≥2 cycles on record
- Quarterly clinical audit of alert accuracy by Clinical Informatics team

---

#### Infrastructure & Security Requirements

- **Deployment environment**: On-premises server within KHCC network perimeter (no cloud PHI export)
- **Authentication**: LDAP integration with KHCC Active Directory; MFA for all clinical users
- **Encryption**: AES-256 at rest; TLS 1.3 in transit
- **Audit logging**: All access and alert generation events logged to tamper-evident store
- **Data retention**: Intake records retained per Jordanian health data regulations (minimum 10 years)
- **Role-based access**: Clinicians (read + acknowledge), Pharmacists (interactions view), Admins (full)

#### Estimated Impact

| Metric | Before | After (projected) |
|--------|--------|--------------------|
| Daily review time | 2–3 hours/day | 15–20 min (verification only) |
| Missed chemo holds | ~3–5/month (estimated) | <1/month |
| Drug interaction detection rate | Variable (human) | ≥95% of catalogued interactions |
| Clerk labor redeployable | 0 FTE | ~0.5 FTE to other tasks |

In [ ]:
# ─────────────────────────────────────────────
# Final Output Summary
# ─────────────────────────────────────────────

print('=' * 65)
print('KHCC PATIENT INTAKE PROCESSING SYSTEM — RUN COMPLETE')
print('=' * 65)
print()
print('Files generated:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname:<40} ({size_kb:.1f} KB)')

print()
print('Sections completed:')
for i, section in enumerate([
    'Setup & Imports',
    'PatientIntake Data Model',
    'Clinical Alert Logic',
    'Patient Dataset (18 patients)',
    'Group-Level Statistical Analysis',
    'CSV Export (3 files)',
    'Simulated Email Summary',
    'Critical Reflection',
    'Stretch Challenge (HTML report + trend analysis + deployment proposal)',
], start=1):
    print(f'  Section {i}: ✅ {section}')

print()
print('GitHub commit suggestions:')
print('  Commit 1: "feat: add PatientIntake dataclass and LabValues model"')
print('  Commit 2: "feat: implement alert logic — lab flags, risk stratification, drug interactions"')
print('  Commit 3: "feat: add 18-patient dataset, group stats, CSV export, and email summary"')
print('  Commit 4: "feat: stretch challenge — HTML report, multi-day trends, deployment proposal"')

KHCC PATIENT INTAKE PROCESSING SYSTEM — RUN COMPLETE

Files generated:
  all_patients.csv                         (11.3 KB)
  cbc_trend_analysis.csv                   (0.7 KB)
  daily_intake_email.txt                   (4.7 KB)
  daily_intake_report.html                 (9.1 KB)
  drug_interaction_alerts.csv              (4.3 KB)
  high_risk_patients.csv                   (6.2 KB)

Sections completed:
  Section 1: ✅ Setup & Imports
  Section 2: ✅ PatientIntake Data Model
  Section 3: ✅ Clinical Alert Logic
  Section 4: ✅ Patient Dataset (18 patients)
  Section 5: ✅ Group-Level Statistical Analysis
  Section 6: ✅ CSV Export (3 files)
  Section 7: ✅ Simulated Email Summary
  Section 8: ✅ Critical Reflection
  Section 9: ✅ Stretch Challenge (HTML report + trend analysis + deployment proposal)

GitHub commit suggestions:
  Commit 1: "feat: add PatientIntake dataclass and LabValues model"
  Commit 2: "feat: implement alert logic — lab flags, risk stratification, drug interactions"
  Commit 